# Retrieval-Augmented Generation (RAG)

RAG combines a **retriever** (search relevant documents) with a **generator** (produce an answer).
This notebook builds a simple RAG pipeline:
1. **Chunking** text into passages
2. **Embedding** with sentence-transformers
3. **Indexing** with FAISS for fast retrieval
4. **Generating** an answer from retrieved context

In [ ]:
import numpy as np
import textwrap

try:
    from sentence_transformers import SentenceTransformer
    HAS_SBERT = True
except ImportError:
    HAS_SBERT = False
    print('sentence-transformers not installed. Using random embeddings as fallback.')
    print('Install with: pip install sentence-transformers')

try:
    import faiss
    HAS_FAISS = True
except ImportError:
    HAS_FAISS = False
    print('FAISS not installed. Using numpy cosine similarity as fallback.')
    print('Install with: pip install faiss-cpu')

np.random.seed(42)
print('Setup complete.')

## 1. Document Chunking

Split a long document into overlapping chunks. Key parameters:
- **chunk_size**: number of characters per chunk
- **overlap**: characters shared between consecutive chunks

In [ ]:
def chunk_text(text, chunk_size=300, overlap=50):
    """Split text into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        # Try to break at sentence boundary
        if end < len(text):
            last_period = chunk.rfind('.')
            if last_period > chunk_size // 2:
                chunk = chunk[:last_period + 1]
                end = start + last_period + 1
        chunks.append(chunk.strip())
        start = end - overlap
    return chunks

# Sample knowledge base
document = """Topological Data Analysis (TDA) uses tools from algebraic topology to study the shape of data. 
The main tool is persistent homology, which tracks topological features (connected components, loops, voids) 
across multiple scales. A key object is the persistence diagram, which summarises the birth and death of 
topological features. TDA has applications in neuroscience, materials science, and genomics. 
The Vietoris-Rips complex is commonly used to build simplicial complexes from point cloud data. 
Mapper is another TDA tool that creates a graph summarising high-dimensional data structure. 
Stability theorems guarantee that small perturbations in data lead to small changes in persistence diagrams. 
Software libraries include Ripser, GUDHI, and giotto-tda for computing persistent homology efficiently. 
Recent advances combine TDA with machine learning, using persistence-based features as input to classifiers. 
The Wasserstein distance between persistence diagrams provides a metric for comparing topological summaries."""

chunks = chunk_text(document, chunk_size=250, overlap=30)
print(f'Document length: {len(document)} chars')
print(f'Number of chunks: {len(chunks)}')
for i, c in enumerate(chunks):
    print(f'\nChunk {i}: ({len(c)} chars)')
    print(textwrap.fill(c, width=80))

## 2. Embedding the Chunks

We encode each chunk into a dense vector using a sentence embedding model.

In [ ]:
if HAS_SBERT:
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(chunks, normalize_embeddings=True)
else:
    # Fallback: random embeddings (for demonstration only)
    embed_dim = 384
    embeddings = np.random.randn(len(chunks), embed_dim).astype('float32')
    # Normalise
    embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

embeddings = np.array(embeddings, dtype='float32')
print(f'Embedding matrix shape: {embeddings.shape}')
print(f'Each chunk is a {embeddings.shape[1]}-dimensional vector.')

## 3. Indexing with FAISS

FAISS provides **fast approximate nearest neighbor** search, essential for large-scale RAG.

In [ ]:
def build_index(embeddings):
    """Build a FAISS index or fallback to numpy."""
    if HAS_FAISS:
        dim = embeddings.shape[1]
        index = faiss.IndexFlatIP(dim)  # inner product (cosine on normalised vectors)
        index.add(embeddings)
        return index
    return None

def retrieve(query_text, index, chunks, embeddings, top_k=3):
    """Retrieve top-k most relevant chunks for a query."""
    # Encode query
    if HAS_SBERT:
        q_emb = model.encode([query_text], normalize_embeddings=True).astype('float32')
    else:
        q_emb = np.random.randn(1, embeddings.shape[1]).astype('float32')
        q_emb = q_emb / np.linalg.norm(q_emb)
    
    if HAS_FAISS and index is not None:
        scores, indices = index.search(q_emb, top_k)
        scores = scores[0]
        indices = indices[0]
    else:
        # Numpy fallback: cosine similarity
        scores = (embeddings @ q_emb.T).flatten()
        indices = np.argsort(scores)[::-1][:top_k]
        scores = scores[indices]
    
    results = []
    for idx, score in zip(indices, scores):
        results.append({'chunk_id': int(idx), 'score': float(score), 'text': chunks[int(idx)]})
    return results

index = build_index(embeddings)

# Test retrieval
query = "What is persistent homology?"
results = retrieve(query, index, chunks, embeddings, top_k=3)

print(f'Query: "{query}"\n')
for r in results:
    print(f'Score: {r["score"]:.4f} | Chunk {r["chunk_id"]}:')
    print(textwrap.fill(r['text'], width=80))
    print()

## 4. Generate an Answer (RAG)

We combine retrieved context with the user's question into a prompt for the LLM.

In [ ]:
def build_rag_prompt(query, retrieved_chunks):
    """Build a RAG prompt with retrieved context."""
    context = "\n\n".join([f"[Source {i+1}] {r['text']}" for i, r in enumerate(retrieved_chunks)])
    prompt = f"""Answer the question based ONLY on the provided context. 
If the context doesn't contain enough information, say "I don't have enough information."

Context:
{context}

Question: {query}

Answer:"""
    return prompt

# Build the prompt
queries = [
    "What is persistent homology?",
    "What software can I use for TDA?",
    "How is TDA used with machine learning?",
]

for q in queries:
    results = retrieve(q, index, chunks, embeddings, top_k=2)
    prompt = build_rag_prompt(q, results)
    print(f'--- Query: {q} ---')
    print(f'Retrieved {len(results)} chunks (best score: {results[0]["score"]:.3f})')
    print(f'Prompt length: {len(prompt)} chars')
    print(f'(In production, this prompt would be sent to the LLM.)\n')

In [ ]:
class SimpleRAG:
    """A minimal RAG pipeline."""
    def __init__(self, documents, chunk_size=250, overlap=30):
        # Chunk
        self.chunks = []
        for doc in documents:
            self.chunks.extend(chunk_text(doc, chunk_size, overlap))
        
        # Embed
        if HAS_SBERT:
            self.embeddings = model.encode(self.chunks, normalize_embeddings=True).astype('float32')
        else:
            self.embeddings = np.random.randn(len(self.chunks), 384).astype('float32')
            self.embeddings /= np.linalg.norm(self.embeddings, axis=1, keepdims=True)
        
        # Index
        self.index = build_index(self.embeddings)
        print(f'RAG index built: {len(self.chunks)} chunks, {self.embeddings.shape[1]}d embeddings')
    
    def query(self, question, top_k=3):
        results = retrieve(question, self.index, self.chunks, self.embeddings, top_k)
        prompt = build_rag_prompt(question, results)
        return {'prompt': prompt, 'sources': results}

# Build pipeline
rag = SimpleRAG([document])

# Query
result = rag.query("What is the Wasserstein distance used for in TDA?")
print(f'\nGenerated prompt ({len(result["prompt"])} chars):')
print(result['prompt'][:500] + '...')

# --- 5. Complete RAG Pipeline ---
class SimpleRAG:
    """A minimal RAG pipeline."""
    def __init__(self, documents, chunk_size=250, overlap=30):
        # Chunk
        self.chunks = []
        for doc in documents:
            self.chunks.extend(chunk_text(doc, chunk_size, overlap))
        
        # Embed
        if HAS_SBERT:
            self.embeddings = model.encode(self.chunks, normalize_embeddings=True).astype('float32')
        else:
            self.embeddings = np.random.randn(len(self.chunks), 384).astype('float32')
            self.embeddings /= np.linalg.norm(self.embeddings, axis=1, keepdims=True)
        
        # Index
        self.index = build_index(self.embeddings)
        print(f'RAG index built: {len(self.chunks)} chunks, {self.embeddings.shape[1]}d embeddings')
    
    def query(self, question, top_k=3):
        results = retrieve(question, self.index, self.chunks, self.embeddings, top_k)
        prompt = build_rag_prompt(question, results)
        return {'prompt': prompt, 'sources': results}

# Build pipeline
rag = SimpleRAG([document])

# Query
result = rag.query("What is the Wasserstein distance used for in TDA?")
print(f'\nGenerated prompt ({len(result["prompt"])} chars):')
print(result['prompt'][:500] + '...')